# 00 — Key Concepts: Can VLMs Read an LA Parking Sign?

This notebook covers the background needed before building anything. It assumes you already know how LLMs work (transformers, tokenization, autoregressive decoding, instruction tuning), so it focuses on what changes when images are added.

**Contents**

*Part 1: VLM fundamentals*
1. What a VLM is
2. How VLMs extend LLMs
3. Vision encoders (ViT, SigLIP)
4. Contrastive image–text pretraining (CLIP / SigLIP)
5. The connector (projector)
6. Visual tokens and the token budget
7. Resolution handling: native resolution vs. dynamic tiling
8. How VLMs are trained
9. OCR inside a VLM vs. a dedicated OCR model
10. Known VLM failure modes

*Part 2: The core research idea*

11. Separating perception from reasoning (conditions A and B)
12. Parking signs as a compositional reasoning task

*Part 3: Inference mechanics*

13. Precision and GPU memory
14. vLLM basics
15. Decoding settings and determinism
16. Structured output (guided JSON decoding)
17. Prompting VLMs

> Architecture details for specific models (patch sizes, tile sizes, token counts) change between releases. They're marked **[VERIFY]** and should be checked against each model's card and `preprocessor_config.json` before you rely on them.

---
# Part 1 — VLM fundamentals

## 1. What a VLM is

A modern open-weights vision-language model (VLM) is an LLM with an extra input path for images. It has three components:

![LLaVA-style VLM architecture](figures/vlm_architecture.svg)

*For comparison, the original LLaVA paper's architecture figure is [here](https://llava-vl.github.io/images/llava_arch.png) (Liu et al., 2023, "Visual Instruction Tuning"). It shows the same idea with a single linear projection as the connector.*

- **Vision encoder**: a Vision Transformer (ViT) that turns pixels into a grid of feature vectors.
- **Connector**: a small network that maps those features into the LLM's embedding space.
- **LLM**: an ordinary decoder-only language model. From its point of view, the image is just a run of extra "tokens" placed in the sequence alongside the text.

After the connector, nothing is special about the image. The LLM attends over visual and text embeddings with the same causal attention it uses for text alone. Everything you know about LLM decoding, context length, KV caches and chat templates still applies.

This design is often called **"LLaVA-style"** after the paper that popularized it. The main alternative, cross-attention between the LLM and a separate image stream (Flamingo, Llama 3.2 Vision), is less common in current open models.

**The three models in this project** all follow the LLaVA-style pattern **[VERIFY]**:

| Model | Vision encoder | LLM backbone | Image handling |
|---|---|---|---|
| Qwen3-VL-8B | ViT (SigLIP-2 initialized), with multi-level features fed into the LLM ("DeepStack") | Qwen3-8B | Native resolution |
| InternVL3.5-8B | InternViT-300M | Qwen3-8B | Dynamic 448-px tiling |
| MiniCPM-V-4.5 | SigLIP-2 (~400M) | Qwen3-8B | Slicing + resampler compression |

All three share essentially the **same LLM backbone family**. That's convenient for this project: differences between them in condition A come mostly from the vision side (encoder, connector, resolution strategy, and vision-language training data), not from differences in the language model's raw reasoning ability. Their post-training still differs, so condition B won't be identical across them.

## 2. How VLMs extend LLMs

A VLM is an LLM with very few changes. It's worth being precise about what changes and what doesn't, because the parts that stay the same (the language model's reasoning) and the parts that are new (reading pixels) are exactly what conditions A and B try to separate.

### A short history

- **2021 — CLIP** showed that images and text can share one embedding space (Section 4).
- **2022 — Flamingo** (DeepMind) connected a frozen vision encoder to a frozen LLM with new cross-attention layers, and showed few-shot visual Q&A.
- **2023 — GPT-4V** made image input a mainstream LLM feature. The same year, **LLaVA** showed a surprisingly simple open recipe: take a CLIP encoder and an open LLM (Vicuna), connect them with a single linear layer, and train on about 600K caption pairs plus GPT-generated visual instruction data. Most open VLMs since then are refinements of that recipe.
- **2024–2025** — higher resolutions, OCR- and document-heavy training data, and reasoning-focused post-training. Open 8B models now score close to closed frontier models on many document and OCR benchmarks.

### What changes, what stays the same

| | Text-only LLM (GPT-style) | VLM |
|---|---|---|
| **Input** | Token IDs | Token IDs **plus** pixel tensors |
| **Turning input into embeddings** | Look up each token ID in the embedding table | Text: same table lookup. Image: vision encoder + connector produce embedding vectors **directly**, skipping the table |
| **Vocabulary** | Fixed discrete vocabulary | Unchanged, except for a few special marker tokens (`<image>`, `<|vision_start|>`). Image "tokens" are continuous vectors, not vocabulary entries |
| **Output** | Text | Still **text only**. These models can't generate images; models that do (e.g. Chameleon) use discrete image tokens instead |
| **Position encoding** | 1D (e.g. RoPE over the sequence index) | Often extended so image tokens carry 2D (height, width) or 3D (time, height, width) positions, e.g. Qwen's multimodal RoPE **[VERIFY per model]** |
| **Attention** | Causal over the whole sequence | Still causal inside the LLM. Image patches were already mixed bidirectionally inside the ViT, so the causal mask costs little |
| **Parameters** | The LLM | LLM + vision encoder (~0.3–0.6B) + connector (tens of millions). An 8B VLM is about 95% language model |
| **Training objective** | Next-token prediction | **The same** next-token prediction, with loss computed only on text tokens. There's no loss for "predicting the image" |
| **Inference** | Prefill the prompt, then decode | Run the vision encoder first, then prefill (much longer, because of the image tokens), then decode exactly as before |

### What a VLM gains

- **Grounding.** Words can refer to things in pixels, so it can describe, locate, compare and count things in an image.
- **Reading.** OCR, charts, tables, screenshots and documents, learned end to end rather than through a separate OCR system.
- **Visual reasoning.** Combining what it sees with world knowledge ("this sign is red, so it's a prohibition").

### What it doesn't gain, and what it can lose

- **Its reasoning is the LLM's reasoning.** Time arithmetic, day-range logic and rule priority all happen in the language model. The vision side supplies a description of the sign and nothing else. That's why condition B, which gives the model a perfect description, measures the reasoning ceiling.
- **Multimodal training can slightly weaken text abilities.** Fine-tuning an LLM on image data sometimes lowers its text-only benchmark scores. Model developers try to prevent this by mixing text-only data into training, with mixed success.
- **New failure modes appear at the boundary.** Two examples: hallucinating visual content, and "answering without looking", where the model answers from language priors while mostly ignoring the image. Several benchmarks have found that VLMs score far above chance on visual questions *with the image removed*. Section 10 covers how this applies to parking signs.

> **A possible reference point for later:** the base text-only Qwen3-8B could also be run on condition B. Comparing it with the three VLMs on B would show whether multimodal training changed their reasoning on this task. That's not in the MVP plan; it's just a cheap extra if there's GPU time left over.

## 3. Vision encoders (ViT, SigLIP)

A **Vision Transformer (ViT)** treats an image the way an LLM treats text:

1. Split the image into a grid of fixed-size **patches** (typically 14×14 or 16×16 pixels).
2. Flatten each patch and linearly project it to a vector, giving one "patch embedding" per patch.
3. Add position information (learned 2D embeddings or 2D RoPE).
4. Run a stack of transformer layers with **bidirectional** attention (every patch sees every other patch; no causal mask).

The output is one feature vector per patch, a spatial grid that roughly preserves where things are in the image.

A 448×448 image with 14-px patches gives a 32×32 = 1,024 patch grid. Because attention cost grows quadratically with patch count, resolution is expensive. Section 7 covers how that cost is managed.

**Why it matters here:** a single character on a sign crop may span only a few patches. Whether the encoder can resolve "8" vs. "3" or "AM" vs. "PM" depends directly on how many pixels reach it, which is set by crop size, resize policy and patch size.

## 4. Contrastive image–text pretraining (CLIP / SigLIP)

Most VLM vision encoders don't start from scratch. They start from a model trained contrastively on hundreds of millions of image–caption pairs from the web.

**CLIP** trains an image encoder and a text encoder together. For a batch of N (image, caption) pairs:
- Embed all N images and all N captions.
- Compute the N×N matrix of cosine similarities.
- Train with a softmax cross-entropy loss (InfoNCE) so that each image's matching caption scores highest in its row, and vice versa.

**SigLIP** replaces the batch-wide softmax with an independent **sigmoid** loss per pair ("is this a match, yes or no?"). That removes the need for global normalization across the batch, so it scales better and tends to give stronger encoders. SigLIP / SigLIP-2 is now the default starting point for open VLMs.

The result is an image encoder whose features are already **aligned with language concepts**.

**Why it matters here:** web captions often mention text that appears in the image, so contrastive encoders learn some reading ability almost by accident. The well-known "typographic attack", where a sticky note reading "iPod" on an apple makes CLIP classify it as an iPod, shows this. The VLM inherits a bias toward text-in-image from the start. Contrastive pretraining alone is weak at *precise* reading, though. Exact reading comes later, from OCR-heavy training data (Section 8).

## 5. The connector (projector)

The vision encoder's features live in a different vector space, and usually a different dimension, from the LLM's token embeddings. The connector bridges them. Common designs:

| Connector | What it does | Output tokens |
|---|---|---|
| **MLP projector** (LLaVA-1.5, InternVL) | 2-layer MLP applied to each patch feature | Same as the number of patches (before any merging) |
| **Patch merging / pixel shuffle + MLP** (Qwen-VL, InternVL) | Concatenate each 2×2 block of neighbouring patches into one vector, then apply the MLP | 4× fewer |
| **Resampler / Q-Former** (BLIP-2, MiniCPM-V) | A fixed set of learned query vectors cross-attends to all patch features | A fixed small number (e.g. 64) regardless of input |

The trade-off is **tokens vs. detail**. Merging and resampling cut sequence length, so inference is cheaper and faster, but they squeeze more of the image into each token. For dense small text like a parking sign, aggressive compression is a plausible cause of misreads. That's one reason the three models may perform differently in condition A even though they share an LLM.

### When does the projection happen?

The connector's weights are **learned during training** (mainly in the alignment stage, §8) and then frozen into the checkpoint. The projection itself **runs at inference time**, as part of the forward pass of every request that contains an image. It isn't a separate preprocessing step:

![One VLM inference request](figures/vlm_inference_pipeline.svg)

The vision encoder and connector run **once per image**, at the start of the request (the prefill phase). While the answer is generated token by token, the visual tokens are already in the KV cache, so the image path sits idle. Since each sign in this project gets 15 queries, vLLM's caching of image encodings and shared prefixes (§14) saves most of the repeated work.

## 6. Visual tokens and the token budget

After the connector, an image is a sequence of **visual tokens**, embedding vectors that go into the LLM just like text token embeddings. They take up context length and KV-cache memory like any other token.

Typical counts **[VERIFY per model]**:
- Qwen-VL family: about one token per 32×32-pixel region after 2×2 merging (Qwen3-VL uses 16-px patches; Qwen2.5-VL used 14-px patches, i.e. 28×28 per token). A 1024×768 image costs about 768 tokens.
- InternVL: 256 tokens per 448×448 tile. A 1024×768 image split into 12 tiles plus a thumbnail costs about 3,300 tokens.
- MiniCPM-V: a resampler compresses each slice to a small fixed count (tens of tokens), so it's very token-efficient.

For comparison, the text part of a prompt in this project (task, query and output instructions) is maybe 150–300 tokens. **The image dominates the cost of every example.**

**Why it matters here:**
- **Cost and throughput.** 7,500 queries × 3 models × condition A means the per-image token count directly sets GPU hours.
- **A fairness question.** If one model sees a sign crop as 3,000 tokens and another as 300, is it the model or the resolution you're comparing? Record tokens-per-image for every model (the plan already asks for this) and report it next to accuracy.
- **An easy optimization.** Each sign is paired with 15 queries. Putting the image *first* in the prompt and the query *last* gives a long shared prefix, which vLLM's prefix caching (Section 14) can reuse across all 15.

The cell below estimates token counts for the two schemes whose rules are simple enough to write down.

In [ ]:
import math

def qwen_style_tokens(w, h, px_per_token=32, min_pixels=256 * 32 * 32, max_pixels=1280 * 32 * 32):
    # Native-resolution scheme: resize so each side is a multiple of px_per_token and
    # total pixels fall within [min_pixels, max_pixels]; one token per px_per_token^2 region.
    # Defaults are illustrative [VERIFY against the model's processor config].
    scale = 1.0
    if w * h > max_pixels:
        scale = math.sqrt(max_pixels / (w * h))
    elif w * h < min_pixels:
        scale = math.sqrt(min_pixels / (w * h))
    rw = max(px_per_token, round(w * scale / px_per_token) * px_per_token)
    rh = max(px_per_token, round(h * scale / px_per_token) * px_per_token)
    return (rw // px_per_token) * (rh // px_per_token), (rw, rh)

def internvl_style_tokens(w, h, tile=448, tokens_per_tile=256, max_tiles=12):
    # Dynamic tiling: choose the tile grid (cols x rows, <= max_tiles) whose aspect ratio is
    # closest to the image's, resize to that grid, and add a thumbnail tile when there is more than one tile.
    # Simplified version of InternVL's find_closest_aspect_ratio [VERIFY].
    aspect = w / h
    grids = [(c, r) for c in range(1, max_tiles + 1) for r in range(1, max_tiles + 1) if c * r <= max_tiles]
    best = min(grids, key=lambda g: (abs(aspect - g[0] / g[1]), -g[0] * g[1]))
    n = best[0] * best[1]
    n_total = n + (1 if n > 1 else 0)
    return n_total * tokens_per_tile, best

for (w, h) in [(300, 450), (600, 900), (1024, 768), (2048, 1536)]:
    q, q_size = qwen_style_tokens(w, h)
    i, grid = internvl_style_tokens(w, h)
    print(f"{w:>4}x{h:<4}  native-res: {q:>5} tokens (resized to {q_size[0]}x{q_size[1]})   "
          f"tiling: {i:>5} tokens (grid {grid[0]}x{grid[1]} + thumb)")

Two things stand out. Under tiling, a small 300×450 crop still costs about 1,800 tokens, because it's upscaled to fill a 2×3 grid of tiles. That's roughly 7× the native-resolution count for the same pixels. Native-resolution cost scales with pixel count, up to its cap. So the **resize policy you pick for crops is an experimental variable**. It should be fixed and documented rather than left to each model's defaults.

## 7. Resolution handling: native resolution vs. dynamic tiling

Early VLMs resized every image to a fixed square (224 or 336 px). That's fine for "is there a dog?" and hopeless for reading small text. Current models use one of two strategies:

**Native (dynamic) resolution** (Qwen-VL family)
- Keep the image close to its original size and aspect ratio (snapped to a multiple of the patch size).
- The ViT processes a variable-length patch sequence, with 2D rotary position embeddings so it knows where each patch is.
- Token count scales with pixel count. `min_pixels` / `max_pixels` settings bound it.

**Dynamic tiling** (InternVL, MiniCPM-V's "slicing")
- The ViT only accepts a fixed input size (e.g. 448×448).
- Choose a grid of tiles that matches the image's aspect ratio, resize the image to fill it, and encode each tile separately.
- Add a downscaled **thumbnail** of the whole image so the model keeps a global view.
- Token count scales with the number of tiles.

**Why it matters here:**
- Tiling can **cut a sign across tile boundaries**, splitting a line of text or separating a panel from its time range. The thumbnail helps, but it's low resolution.
- Upscaling a small crop doesn't add information, but it does add tokens.
- The plan's instruction to "resize crops to a sensible max dimension and log tokens-per-image" follows from this. Pick one policy (for example, longest side 1024 px) and apply it before any model's processor sees the image.

## 8. How VLMs are trained

Most current open VLMs go through roughly three stages:

1. **Alignment pretraining.** Freeze the vision encoder and the LLM; train only the connector on image–caption pairs. This teaches the connector to "translate" vision features into something the LLM can read. It's cheap and fast.
2. **Multimodal pretraining / continued training.** Unfreeze more (often everything) and train on a large mix: interleaved image-text web documents, captions, **OCR and document data** (scanned pages, charts, screenshots, synthetic rendered text), grounding data, and video.
3. **Instruction tuning and post-training.** Supervised fine-tuning on visual Q&A and chat data, then often preference optimization or RL. Recent releases (Qwen3-VL, InternVL3.5, MiniCPM-V 4.5) also ship "thinking" variants or modes trained to produce long reasoning chains before answering.

**Why it matters here:**
- **Reading skill comes from the data mix.** A model's OCR accuracy on street signs depends heavily on how much scene-text and document data went into stage 2. Model cards usually report OCR benchmarks (OCRBench, TextVQA, DocVQA). These are a useful prior, but they're mostly documents and product photos, not angled, weathered street signs.
- **Thinking vs. non-thinking is a hidden variable.** If one model runs with reasoning enabled and another doesn't, condition A/B differences partly reflect that. Pick one mode (the plan implies instruct / non-thinking), use it for all three models, and write it down.

## 9. OCR inside a VLM vs. a dedicated OCR model

**Dedicated OCR** (PaddleOCR, EasyOCR) is a pipeline: a detection model finds text boxes, then a recognition model reads the characters in each box. The output is strings plus bounding boxes plus confidence scores. It's fast, cheap and good at "what characters are here?", but it has no idea what the text *means* or how lines relate to each other.

**A VLM has no OCR module.** Reading is an emergent skill of the encoder + connector + LLM, learned end to end from data. That has consequences:
- It can use **context** to recover degraded text (knowing "STREET CLEANING" is a common phrase helps it fill in a smudged letter). That's a strength.
- The same context can **override the pixels** (it "reads" what it expects to see). That's a failure mode (Section 10).
- It doesn't naturally report per-character confidence or bounding boxes.
- Reading and reasoning happen in one forward pass, so you can't tell from the output alone which one failed. Condition B exists to solve exactly this (Section 11).

**In this project**, dedicated OCR is only a **cheap filter** in the collection funnel ("does this image contain parking vocabulary?"). It isn't a baseline and isn't in the evaluation. An OCR-then-LLM baseline is one of the deferred extensions.

## 10. Known VLM failure modes

These are the failure types you should expect, and that the error taxonomy later in the project is designed to catch:

| Failure mode | What it looks like on a parking sign |
|---|---|
| **Fine-grained misreads** | 8↔3, 1↔7, 6↔8, "AM"↔"PM", "MON"↔"MOM", and similar digit/letter confusions at low resolution |
| **Language-prior override** | Reads "2 HR" where the sign says "1 HR" because "2 HR PARKING" is more common; assumes typical hours instead of reading them |
| **Hallucinated content** | Invents a panel, an exception ("EXCEPT SUNDAY") or a permit district that isn't there |
| **Dropped content** | Ignores a panel, especially lower or smaller panels in a tall stack |
| **Spatial / ordering errors** | Attaches a time range to the wrong panel, or mixes up which rule goes with which line |
| **Counting** | Miscounts the panels on a sign (VLMs are notoriously weak at counting) |
| **Robustness** | Degrades with oblique angles, glare, occlusion (tree branches, other signs), panorama distortion |
| **Reasoning errors** (even with correct reading) | Wrong time arithmetic, overnight windows (6 PM–8 AM), day ranges (MON–FRI includes Wednesday), priority between conflicting panels, duration vs. time limit |
| **Sycophancy / query leakage** | The phrasing of the query ("I just need 10 minutes, it's fine right?") nudges the verdict |

The first seven are **perception** failures; the last two are **reasoning or prompting** failures. Being able to split them apart is the point of the experimental design.

---
# Part 2 — The core research idea

## 11. Separating perception from reasoning (conditions A and B)

The research question has two parts: can a VLM get the verdict right, and **when it fails, is it because it couldn't read the sign or couldn't reason about it?**

End-to-end accuracy alone can't answer the second part. The design uses two conditions per model:

| Condition | Input | What must go right |
|---|---|---|
| **A** (image) | Sign crop + query | Perception **and** reasoning |
| **B** (text) | Ground-truth transcription of the sign + query | Reasoning only (perception is handed over for free) |

Then:

- **Accuracy(B)** ≈ the model's reasoning ceiling on this task.
- **Accuracy(B) − Accuracy(A)** ≈ the **perception cost**, i.e. how much accuracy is lost by making the model read the sign itself.

If a model scores 90% in B and 60% in A, most of its failures are reading failures. If it scores 65% in B and 60% in A, the bottleneck is reasoning, and a better vision encoder wouldn't help much. Plotting both against panel count shows whether complexity hurts reading, reasoning, or both.

**Caveats worth knowing from the start** (they belong in the limitations section):
- **B isn't "perfect perception" in a neutral sense.** The ground truth is given in *your* structured format, already parsed into panels, days as lists, and 24-hour times. That's easier than reading and also easier than a raw text transcription would be. So the gap measures "perception + parsing into structure", not pure perception. The format of B's text is a design choice; decide it once and document it.
- **The gap isn't strictly additive.** A model might reason correctly from a slightly wrong reading, or get lucky from a wrong one. The error review on condition-A failures is how you check what's actually going on.
- **A can occasionally beat B.** The image carries layout cues (panel color, arrows, grouping) that the transcription might flatten.
- **Condition C** (the model transcribes, then reasons over its own transcription) would split the pipeline further. It's deferred to the extensions.

## 12. Parking signs as a compositional reasoning task

An LA parking sign is usually a **vertical stack of panels** on a single pole. Each panel is one rule with conditions:

```
  ┌──────────────────────────────┐
  │   TOW-AWAY  NO STOPPING      │  ← red: prohibition, peak-hour
  │   7AM–9AM  4PM–7PM  MON–FRI  │
  ├──────────────────────────────┤
  │   NO PARKING                 │  ← red: prohibition, weekly
  │   8AM–10AM TUESDAY           │
  │   STREET CLEANING            │
  ├──────────────────────────────┤
  │   2 HR PARKING               │  ← green: time limit
  │   8AM–6PM  EXCEPT SUNDAY     │
  ├──────────────────────────────┤
  │   PERMIT PARKING DISTRICT 7  │  ← exemption from the limit above
  │   PERMIT HOLDERS EXEMPT      │
  └──────────────────────────────┘
```

To answer "Tuesday 11:30, no permit, staying 3 hours — legal?", the model has to:

1. **Read** every panel correctly (text, times, days, AM/PM).
2. **Find which panels apply** at that day and time: a day-range check, a time-window check, sometimes an overnight wraparound.
3. **Resolve conflicts** between applicable panels. The most restrictive one governs (no stopping > no parking > permit-only without a permit > time limit exceeded > legal).
4. **Apply the query's conditions**: permit or no permit, and duration vs. the time limit (for time-limited panels, the question is whether the stay *exceeds* the limit within the window).
5. Output a verdict, ideally naming the governing panel.

This task works well as a benchmark because:
- **Complexity is controllable and measurable.** More panels mean more rules to read and more interactions to resolve, so **panel count** is a natural difficulty axis (the x-axis of the main figure).
- **Ground truth is exact.** Once a sign is transcribed, a small deterministic evaluator gives the correct answer for *any* query. Generating 15 queries per sign is free and needs no extra labeling.
- **It's realistic and has stakes.** Drivers, and increasingly driving systems, face exactly this problem, and existing work on parking signs stops at detection and OCR.
- **It includes classic hard cases:** boundary times (09:59 vs. 10:00), overnight windows, day ranges and exceptions, all of which LLMs are known to handle inconsistently.

---
# Part 3 — Inference mechanics

## 13. Precision and GPU memory

GPU memory needs at inference come from three things:

1. **Weights**: parameters × bytes per parameter. bf16 is 2 bytes, so an 8B model needs about 16 GB. The vision encoder adds a few hundred million parameters, well under 1 GB.
2. **KV cache**: for every token in every sequence in the batch, each layer stores a key and a value vector. Per token, that's `2 × layers × kv_heads × head_dim × bytes`. Grouped-query attention (few KV heads) keeps this small.
3. **Activations and overhead**: CUDA context, temporary buffers, and the vision encoder's activations on large images.

**bf16** (bfloat16) keeps fp32's exponent range with less mantissa precision, which makes it the standard precision for inference. Quantization (8-bit, 4-bit) would cut memory further, but it could also change accuracy. It's not needed for 8B models on a 24–48 GB card, so keep everything in bf16 so results are comparable and clean.

The cell below does the arithmetic for a Qwen3-8B-sized backbone.

In [ ]:
# Qwen3-8B-like backbone config [VERIFY against the model's config.json]
params_b    = 8.2      # billions of parameters (LLM + vision encoder, approx.)
n_layers    = 36
n_kv_heads  = 8        # grouped-query attention
head_dim    = 128
bytes_bf16  = 2

weights_gb = params_b * 1e9 * bytes_bf16 / 1e9
kv_per_token_bytes = 2 * n_layers * n_kv_heads * head_dim * bytes_bf16

print(f"Weights (bf16):      {weights_gb:.1f} GB")
print(f"KV cache per token:  {kv_per_token_bytes / 1024:.0f} KiB")

tokens_per_example = 1000 + 300   # ~1000 visual tokens + ~300 text/prompt/output tokens
for gpu, mem_gb in [("RTX 4090", 24), ("L40S", 48), ("A100 80GB", 80)]:
    usable = mem_gb * 0.9 - weights_gb - 2        # vLLM default uses ~90% of memory; ~2 GB overhead
    n_tokens = usable * 1e9 / kv_per_token_bytes
    print(f"{gpu:>10}: ~{usable:4.1f} GB for KV cache -> ~{n_tokens/1e3:,.0f}k tokens "
          f"-> ~{n_tokens / tokens_per_example:,.0f} concurrent examples")

Even a 24 GB card holds dozens of examples in flight at once. That's why the plan says an A100 is overkill for this stage. The larger card mostly buys bigger batches and a bit more speed, not the ability to run at all.

## 14. vLLM basics

A naive Hugging Face `model.generate()` loop processes a fixed batch, pads every sequence to the longest one, and waits for the slowest sequence before starting the next batch. That wastes most of the GPU. **vLLM** is an inference server built to avoid this:

- **PagedAttention.** The KV cache is stored in fixed-size blocks, like OS memory pages, instead of one contiguous buffer per sequence. Almost no memory is wasted on padding or reserved-but-unused space, so many more sequences fit at once.
- **Continuous batching.** Sequences join and leave the running batch at every decoding step. As soon as one finishes, a waiting request takes its slot, so the GPU stays busy.
- **Prefix caching.** Requests that share a prompt prefix reuse the cached KV blocks for that prefix. With 15 queries per sign, putting the image and instructions first means the image is encoded and prefilled once and reused 14 times.
- **Multimodal support.** vLLM runs each model's own image processor and vision encoder, so you pass images alongside the chat messages and it handles tiling and resizing according to the model's config (or your overrides).
- **Offline or server mode.** You can call `LLM.generate()` / `LLM.chat()` directly in a script (simplest for a batch job like this) or run an OpenAI-compatible HTTP server.

For this project, throughput is 10–20× that of a naive loop, which is the difference between a few GPU-hours and a few GPU-days.

## 15. Decoding settings and determinism

For a benchmark you want answers to be **reproducible** and to reflect the model's **most likely** output, not a random sample.

- **`temperature=0`** means greedy decoding: always pick the highest-probability token. vLLM treats temperature 0 as greedy.
- **A fixed seed** only matters if you sample (temperature > 0). Set it anyway so any sampled run is reproducible.
- **Keep `max_tokens` tight.** A JSON verdict needs perhaps 50–150 tokens. A tight cap stops runaway generations from wasting compute.

**Greedy isn't perfectly deterministic on GPUs.** Floating-point addition isn't associative, and the order of operations inside GPU kernels can depend on batch size and composition. So the same prompt can occasionally produce a different token when it's batched with different neighbours, especially when two candidate tokens are nearly tied. In practice this affects a small fraction of outputs. The mitigations are:
- Cache every result keyed on `(model, image hash, prompt hash, settings)`, as the plan specifies, so that once an answer exists it's fixed.
- Record the vLLM version, model revision and GPU type with the results.
- Optionally rerun a small subset to measure the flip rate, and report it if it's non-trivial.

## 16. Structured output (guided JSON decoding)

Parsing free-text answers ("Well, it depends… I'd say you're probably fine") is fragile and introduces your own parser errors into the results. **Guided decoding** removes the problem.

**How it works:** you give the server a JSON Schema (generated from a Pydantic model). At each decoding step, the engine works out which tokens could still lead to schema-valid JSON and sets the probability of every other token to zero before picking. The output is **guaranteed to parse**. vLLM supports this through backends such as xgrammar and outlines (the API parameter name has changed between versions **[VERIFY]**).

A response schema for this task might look like:

```python
class Answer(BaseModel):
    reasoning: str                                    # optional; see below
    governing_panel: int | None                       # which panel decides it
    verdict: Literal["legal", "illegal", "ambiguous"]
```

**Subtleties that affect results:**
- **Field order matters.** Decoding is autoregressive. If `verdict` comes first, the model commits to an answer before writing any reasoning; if `reasoning` comes first, it gets to "think" before committing. That's a real experimental choice that changes accuracy. Make it deliberately and keep it the same across models.
- **Constraints can distort outputs.** If the model's preferred next token is disallowed, it has to take a lower-probability path, which occasionally produces odd content that is still valid JSON. Small schemas with clear field names and descriptions keep this rare.
- **Parse failures aren't exactly zero.** Hitting `max_tokens` in the middle of the JSON truncates it. Log and count these; the plan's gate is under 1%.
- **Tell the model about the schema in the prompt as well.** Guided decoding enforces the format, but the model answers better when the prompt also describes the expected fields.

## 17. Prompting VLMs

Prompting a VLM is prompting an LLM, with a few extra considerations:

- **Chat templates and image placeholders.** Each model has its own chat template and its own image placeholder token(s) (`<image>`, `<|vision_start|>…<|vision_end|>`, etc.). Always build prompts through the model's processor or vLLM's chat interface (`messages` with `{"type": "image"}` entries), never by hand-concatenating strings. A wrong template silently degrades accuracy.
- **Image placement.** Putting the image **before** the question is the convention most models were trained on, and it also enables prefix caching across the 15 queries per sign.
- **One fixed prompt across models.** The plan deliberately uses one prompt: a task statement, the query, and the output schema instructions. Per-model prompt tuning would turn the comparison into a prompt-engineering contest. Some models are known to be sensitive to prompt wording, so a small sensitivity check on a few paraphrases is a useful robustness note later.
- **Conditions A and B should share the same prompt**, with only the sign representation swapped (image vs. transcription). Any other difference between them contaminates the A−B gap.
- **No domain primer.** The prompt doesn't explain LA sign conventions (e.g. "a time limit panel below a permit panel means…"). Whether a primer helps is one of the extension experiments, so leaving it out keeps the MVP clean.
- **Thinking modes.** As noted in Section 8, disable or fix reasoning modes consistently. Check each model's chat template for flags like `enable_thinking`.

---
## Summary: what these concepts imply for the build

| Concept | Concrete consequence for the project |
|---|---|
| VLMs extend LLMs (§2) | Reasoning happens in the LLM, so condition B measures the reasoning ceiling; optionally add text-only Qwen3-8B as a reference on B |
| Visual tokens and resolution (§6–7) | Fix one crop-resize policy for all models; log tokens-per-image per model |
| Training data and thinking modes (§8) | Use instruct / non-thinking mode for all three models and record it |
| OCR vs. VLM (§9) | PaddleOCR is only a collection filter, not an evaluated system |
| Failure modes (§10) | These become the tags in the later error taxonomy |
| Conditions A/B (§11) | Same prompt, only the sign representation swapped; document B's transcription format |
| Compositional structure (§12) | Panel count is the difficulty axis; a deterministic evaluator supplies ground truth |
| Memory (§13) | bf16, no quantization; a 24–48 GB GPU is enough |
| vLLM (§14) | Offline batch mode; image first in the prompt for prefix caching |
| Determinism (§15) | temperature 0, cache results, record versions |
| Guided JSON (§16) | Decide field order (reasoning before verdict?) deliberately; count truncations |
| Prompting (§17) | Use chat templates via the processor; no primer; one prompt for all models |